Задача 1 - реализация series test, рассматриваю размерности d от 1 до 7, для большей - слишком мало данных

In [3]:
from scipy import stats
import numpy as np
import math
import itertools

def rng(m=2**32, a=1103515245, c=12345):
    rng.current = (a * rng.current + c) % m
    return rng.current / m

rng.current = 1
samples = np.array([rng() for _ in range(1000)])

def series_test(samples):
    N = len(samples)
    
    for d in range(1, 5):
        # Trim to multiple of d
        Nd = (N // d)
        samples_trim = samples[:(Nd * d)]
        points = samples_trim.reshape(-1, d)
        num_points = len(points) # соединяем в d - мерные точки
        k = int((num_points // 6)**(1/d)) # делаем такое k, что в каждой E_i > 5 для всех i
        num_bins = k ** d
        bins = [np.linspace(0, 1, k + 1) for _ in range(d)]
        f_obs, edges = np.histogramdd(points, bins=bins)
        f_obs = f_obs.flatten()
        f_exp = np.full(num_bins, num_points / num_bins)
        res = stats.chisquare(f_obs, f_exp)
        p = res.pvalue
        if p > 0.05:
            print(f"d = {d}: passed, pvalue = {p}")
        else:
            print(f"d = {d}: failed, pvalue = {p}")

series_test(samples)

d = 1: passed, pvalue = 0.6589920080022522
d = 2: passed, pvalue = 0.21516098226090657
d = 3: passed, pvalue = 0.07268438663014308
d = 4: passed, pvalue = 0.5110200984502635


Задача 2: permutation test

In [12]:
import math

def perm_rank(perm): # присвоим каждой перестановке ее номер(лексикографический)
    n = len(perm)
    perm = [p - 1 for p in perm]
    rank = 0
    for i in range(n):
        count = 0
        for j in range(i + 1, n):
            if perm[j] < perm[i]:
                count += 1
        rank += count * math.factorial(n - i - 1)
    return rank

def sorted_perm(arr):
    n = len(arr)
    perm = [i for i in range(0, n)]
    for i in range(n):
        swapped = False
        for j in range(0, n - i - 1):
            if arr[j] > arr[j + 1]:
                arr[j], arr[j + 1] = arr[j + 1], arr[j]
                perm[j], perm[j + 1] = perm[j + 1], perm[j]
                swapped = True
        if not swapped:
            break
    return perm

def permutation_test(samples):
    N = len(samples)
    
    for d in range(1, 100):
        # Trim to multiple of d
        Nd = (N // d)
        samples_trim = samples[:(Nd * d)]
        points = samples_trim.reshape(-1, d)
        num_points = len(points)
        perm_indexes = np.array([perm_rank(sorted_perm(list(x))) for x in points])
        counts = {}
        for i in perm_indexes:
            if i in counts.keys():
                counts[i] += 1
            else:
                counts[i] = 1
        unique_numbers, f_obs = np.unique(perm_indexes, return_counts=True)
        f_exp = np.full(len(counts), len(perm_indexes) / len(unique_numbers))
        p = stats.chisquare(f_obs, f_exp).pvalue
        if (p < 0.05):
            print(f'failed at d = {d}')
    else:
        print('never failed!')
permutation_test(samples)

never failed!
